In [1]:
import os
import re
import time
import tempfile
import requests
import numpy as np
import fitz
import arxiv
from collections import Counter
from typing import List, Optional
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from huggingface_hub import InferenceClient
from neo4j import GraphDatabase
from unstructured.partition.pdf import partition_pdf

load_dotenv()

c:\Users\Shreyaan\Desktop\coding\python\aiml\llm\graph-RAG\PaperGraph\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_AUTH     = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
HF_TOKEN       = os.getenv("HF_TOKEN")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

# HuggingFace Inference API — 768-dim to match the Neo4j vector index
# BAAI/bge-base-en-v1.5 : 768-dim, strong retrieval quality
HF_EMBED_MODEL = "BAAI/bge-base-en-v1.5"
EMBED_DIM      = 768
hf_client      = InferenceClient(token=HF_TOKEN)

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
driver.verify_connectivity()

print("Config loaded:")
print(f"  GOOGLE_API_KEY : {'set' if GOOGLE_API_KEY else 'MISSING'}")
print(f"  HF_TOKEN       : {'set' if HF_TOKEN else 'MISSING'}")
print(f"  NEO4J_URI      : {NEO4J_URI}")
print(f"  NEO4J_DATABASE : {NEO4J_DATABASE}")
print(f"  embed model    : {HF_EMBED_MODEL}  (dim={EMBED_DIM})")
print("Neo4j connected.")

Config loaded:
  GOOGLE_API_KEY : set
  HF_TOKEN       : set
  NEO4J_URI      : neo4j+s://f0a68b38.databases.neo4j.io
  NEO4J_DATABASE : f0a68b38
  embed model    : BAAI/bge-base-en-v1.5  (dim=768)
Neo4j connected.


In [3]:
SETUP_QUERIES = [
    # ── Unique constraints ─────────────────────────────────────────────────
    "CREATE CONSTRAINT paper_arxiv_id  IF NOT EXISTS FOR (p:Paper)   REQUIRE p.arxiv_id    IS UNIQUE",
    "CREATE CONSTRAINT author_id       IF NOT EXISTS FOR (a:Author)  REQUIRE a.author_id   IS UNIQUE",
    "CREATE CONSTRAINT section_id      IF NOT EXISTS FOR (s:Section) REQUIRE s.section_id  IS UNIQUE",
    "CREATE CONSTRAINT concept_name    IF NOT EXISTS FOR (c:Concept) REQUIRE c.name_norm   IS UNIQUE",
    "CREATE CONSTRAINT method_name     IF NOT EXISTS FOR (m:Method)  REQUIRE m.name_norm   IS UNIQUE",
    "CREATE CONSTRAINT dataset_name    IF NOT EXISTS FOR (d:Dataset) REQUIRE d.name_norm   IS UNIQUE",
    "CREATE CONSTRAINT topic_name      IF NOT EXISTS FOR (t:Topic)   REQUIRE t.name_norm   IS UNIQUE",
    # ── Vector index (RAG) — dimensions must match EMBED_DIM ───────────────
    (
        "CREATE VECTOR INDEX section_embedding IF NOT EXISTS "
        "FOR (s:Section) ON s.embedding "
        f"OPTIONS {{ indexConfig: {{ `vector.dimensions`: {EMBED_DIM}, `vector.similarity_function`: 'cosine' }} }}"
    ),
    # ── Regular indexes (query performance) ────────────────────────────────
    "CREATE INDEX paper_year   IF NOT EXISTS FOR (p:Paper)  ON (p.year)",
    "CREATE INDEX paper_stub   IF NOT EXISTS FOR (p:Paper)  ON (p.stub)",
    "CREATE INDEX paper_cat    IF NOT EXISTS FOR (p:Paper)  ON (p.category)",
    "CREATE INDEX author_orcid IF NOT EXISTS FOR (a:Author) ON (a.orcid)",
]

for q in SETUP_QUERIES:
    try:
        driver.execute_query(q, database_=NEO4J_DATABASE)
        print(f"  OK  : {q[:80]}")
    except Exception as e:
        print(f"  Skip: {e}")

  OK  : CREATE CONSTRAINT paper_arxiv_id  IF NOT EXISTS FOR (p:Paper)   REQUIRE p.arxiv_
  OK  : CREATE CONSTRAINT author_id       IF NOT EXISTS FOR (a:Author)  REQUIRE a.author
  OK  : CREATE CONSTRAINT section_id      IF NOT EXISTS FOR (s:Section) REQUIRE s.sectio
  OK  : CREATE CONSTRAINT concept_name    IF NOT EXISTS FOR (c:Concept) REQUIRE c.name_n
  OK  : CREATE CONSTRAINT method_name     IF NOT EXISTS FOR (m:Method)  REQUIRE m.name_n
  OK  : CREATE CONSTRAINT dataset_name    IF NOT EXISTS FOR (d:Dataset) REQUIRE d.name_n
  OK  : CREATE CONSTRAINT topic_name      IF NOT EXISTS FOR (t:Topic)   REQUIRE t.name_n
  OK  : CREATE VECTOR INDEX section_embedding IF NOT EXISTS FOR (s:Section) ON s.embeddi
  OK  : CREATE INDEX paper_year   IF NOT EXISTS FOR (p:Paper)  ON (p.year)
  OK  : CREATE INDEX paper_stub   IF NOT EXISTS FOR (p:Paper)  ON (p.stub)
  OK  : CREATE INDEX paper_cat    IF NOT EXISTS FOR (p:Paper)  ON (p.category)
  OK  : CREATE INDEX author_orcid IF NOT EXISTS FOR (a:Auth

In [4]:
class ExtractedEntity(BaseModel):
    name: str = Field(description="Concise name (3-6 words max)")
    description: Optional[str] = Field(None, description="Brief description (optional)")
    confidence: float = Field(
        default=1.0,
        description="How confident you are this entity truly belongs here (0.0-1.0)",
    )


class TopicEntity(BaseModel):
    name: str = Field(description="Topic name (3-6 words max)")
    description: Optional[str] = None
    parent_topic: Optional[str] = Field(
        None,
        description="The broader topic this belongs to, e.g. 'Transformers' -> 'Deep Learning'",
    )
    confidence: float = Field(default=1.0, description="Confidence score 0.0-1.0")


class ConceptPair(BaseModel):
    concept_a: str = Field(description="First concept name")
    concept_b: str = Field(description="Second concept name (closely related to concept_a)")
    confidence: float = Field(default=1.0, description="Confidence score 0.0-1.0")


class MethodExtension(BaseModel):
    """e.g. LoRA extends PEFT, PEFT extends Transformer fine-tuning"""
    child_method: str = Field(description="The more specific / derived method")
    parent_method: str = Field(description="The broader method it builds on")
    confidence: float = Field(default=1.0, description="Confidence score 0.0-1.0")


class PaperExtraction(BaseModel):
    paper_summary: str = Field(
        default="",
        description="2-3 sentence plain-English summary of the paper's contribution",
    )
    concepts_introduced: List[ExtractedEntity] = Field(
        default_factory=list,
        description="NEW concepts/frameworks/terms this paper PROPOSES",
    )
    concepts_applied: List[ExtractedEntity] = Field(
        default_factory=list,
        description="EXISTING concepts or theories this paper USES",
    )
    methods: List[ExtractedEntity] = Field(
        default_factory=list,
        description="Algorithms, models, or techniques (e.g. BERT, gradient descent)",
    )
    datasets: List[ExtractedEntity] = Field(
        default_factory=list,
        description="Datasets used for training or evaluation (e.g. ImageNet, SQuAD)",
    )
    topics: List[TopicEntity] = Field(
        default_factory=list,
        description="Broad research areas, each optionally with a parent_topic",
    )
    concept_relations: List[ConceptPair] = Field(
        default_factory=list,
        description="Pairs of concepts from this paper that are closely related",
    )
    method_extensions: List[MethodExtension] = Field(
        default_factory=list,
        description="Method hierarchy pairs: child_method EXTENDS parent_method",
    )
    extends_papers: List[str] = Field(
        default_factory=list,
        description="Exact titles of prior papers this paper directly builds upon",
    )
    contradicts_papers: List[str] = Field(
        default_factory=list,
        description="Exact titles of prior papers whose findings this paper disputes",
    )


structured_llm = llm.with_structured_output(PaperExtraction)

In [5]:
def fetch_arxiv_papers(query: str, max_results: int = 2) -> list[dict]:
    """Fetch paper metadata from ArXiv and return plain dicts."""
    print(f"[*] Searching ArXiv for: {query}")
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance,
    )
    papers = []
    for result in client.results(search):
        arxiv_id = result.entry_id.split("/abs/")[-1]
        arxiv_id = re.sub(r"v\d+$", "", arxiv_id)   # strip version suffix
        papers.append({
            "arxiv_id"  : arxiv_id,
            "title"     : result.title,
            "abstract"  : result.summary.replace("\n", " "),
            "authors"   : [a.name for a in result.authors],
            "published" : result.published.strftime("%Y-%m-%d"),
            "year"      : result.published.year,
            "pdf_url"   : result.pdf_url,
            "category"  : result.categories[0] if result.categories else "",
        })
        print(f"  [{arxiv_id}] {result.title[:70]}")
    return papers

In [6]:
def download_pdf(pdf_url: str, timeout: int = 30) -> bytes:
    resp = requests.get(pdf_url, timeout=timeout, headers={"User-Agent": "PaperGraph/2.0"})
    resp.raise_for_status()
    return resp.content


# ── unstructured section parsing ─────────────────────────────────────────

_CONTENT_TYPES = {
    "NarrativeText", "Text", "UncategorizedText",
    "ListItem", "FigureCaption", "Table",
}



# ── citation extraction via unstructured elements ────────────────────────

_REF_HEADERS = {"references", "bibliography", "reference list", "works cited"}

_ARXIV_PATS = [
    r"arXiv[:\s]+(\d{4}\.\d{4,5})(?:v\d+)?",       # arXiv:2203.08975
    r"arxiv\.org/(?:abs|pdf)/(\d{4}\.\d{4,5})",     # arxiv.org/abs/...
    r"\[(\d{4}\.\d{4,5})\]",                         # [2203.08975] inline
]

def _arxiv_id(text: str) -> str | None:
    for pat in _ARXIV_PATS:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            return re.sub(r'v\d+$', '', m.group(1))
    return None


def _title_from_entry(text: str) -> str | None:
    """
    Extract a paper title from a single, already-cleaned reference string.
    unstructured gives us one entry per element, so the text is short and
    well-structured — we only need a few targeted patterns.
    """
    # 1. Text in "quotes" or "curly quotes"
    m = re.search(r'["\u201c\u201e]([^"\u201c\u201d\u201e]{15,180})["\u201d\u201e]', text)
    if m:
        return m.group(1).strip()

    # 2. (YEAR). Title.   — APA style
    m = re.search(r'\(\d{4}\)[.,]?\s+([A-Z][^\n]{20,180}?)(?:\.\s+[A-Z]|\.$|,\s+In\b|,\s+Proc)', text)
    if m:
        return m.group(1).strip()

    # 3. YEAR. Title.  — IEEE / Vancouver numbered
    m = re.search(r'\b(?:19|20)\d{2}[.,]\s+([A-Z][^\n]{20,180}?)(?:\.\s+[A-Z]|\.$|,\s+In\b)', text)
    if m:
        return m.group(1).strip()

    # 4. Strip leading "Lastname, F., Lastname, F. (YEAR)" then grab first long phrase
    stripped = re.sub(
        r'^(?:[A-Z][a-zÀ-ÿ\-\u2019]+,?\s+(?:[A-Z]\.?\s*){1,3}(?:and\s+|,\s+)?){1,6}'
        r'(?:\(\d{4}\)\.?\s+)?',
        '', text
    ).strip().lstrip('.,;')
    m = re.match(r'([A-Z][^\n.!?]{20,180})[.!?]', stripped)
    if m:
        cand = m.group(1).strip()
        if len(cand.split()) >= 4:
            return cand

    return None


def partition_pdf_elements(pdf_bytes: bytes) -> list:
    """Write PDF bytes to a temp file and return unstructured elements."""
    with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
        tmp.write(pdf_bytes)
        tmp_path = tmp.name
    try:
        return partition_pdf(filename=tmp_path, strategy="fast",
                             infer_table_structure=False)
    finally:
        os.unlink(tmp_path)


def parse_sections(pdf_bytes: bytes, max_sections: int = 15) -> tuple[list, list]:
    """
    Single-pass parse: returns (sections, cited_refs) from one partition_pdf call.

    sections   : list of {title, content, order}
    cited_refs : list of {arxiv_id: str|None, title: str|None}
    """
    elements = partition_pdf_elements(pdf_bytes)

    # ── Section extraction ────────────────────────────────────────────────
    sections: list = []
    current_title: str | None = None
    current_parts: list = []
    order = 0
    in_references = False

    def flush():
        nonlocal order
        if current_title and current_parts and not in_references:
            content = re.sub(r'\s+', ' ', ' '.join(current_parts)).strip()
            if len(content) > 100:
                sections.append({
                    'title':   current_title,
                    'content': content[:3000],
                    'order':   order,
                })
                order += 1

    ref_elements: list = []

    for el in elements:
        el_type = type(el).__name__
        el_text = (el.text or "").strip()
        if not el_text:
            continue

        if el_type == "Title":
            if el_text.lower().strip(".:") in _REF_HEADERS:
                flush()
                in_references = True
                continue
            if not in_references:
                flush()
                current_title = el_text
                current_parts = []
        elif in_references:
            ref_elements.append(el_text)
        elif el_type in _CONTENT_TYPES:
            if current_title is None:
                current_title = "Introduction"
            current_parts.append(el_text)

    if not in_references:
        flush()   # flush last section when no explicit references header found

    # Fallback chunking if no titled sections found
    if not sections:
        all_words = " ".join(
            (el.text or "") for el in elements
            if type(el).__name__ in _CONTENT_TYPES
        ).split()
        chunk_sz = 500
        sections = [
            {
                'title':   f'Chunk {i // chunk_sz + 1}',
                'content': ' '.join(all_words[i:i + chunk_sz]),
                'order':   i // chunk_sz,
            }
            for i in range(0, min(len(all_words), chunk_sz * max_sections), chunk_sz)
            if ' '.join(all_words[i:i + chunk_sz]).strip()
        ]

    # ── Citation extraction from ref_elements ────────────────────────────
    cited_refs: list = []
    seen_arxiv: set = set()
    seen_titles: set = set()

    for entry in ref_elements:
        if len(entry) < 20:
            continue
        aid = _arxiv_id(entry)
        if aid:
            if aid not in seen_arxiv:
                seen_arxiv.add(aid)
                # still try to extract title from this entry
                t = _title_from_entry(entry)
                cited_refs.append({"arxiv_id": aid, "title": t})
        else:
            t = _title_from_entry(entry)
            if t:
                tnorm = t.lower().strip()
                if tnorm not in seen_titles and len(tnorm.split()) >= 4:
                    seen_titles.add(tnorm)
                    cited_refs.append({"arxiv_id": None, "title": t})

    # Fallback: scan full ref text for any stray arxiv IDs we may have missed
    full_ref_text = "\n".join(ref_elements)
    for pat in _ARXIV_PATS:
        for m in re.finditer(pat, full_ref_text, re.IGNORECASE):
            aid = re.sub(r'v\d+$', '', m.group(1))
            if aid not in seen_arxiv:
                seen_arxiv.add(aid)
                cited_refs.append({"arxiv_id": aid, "title": None})

    return sections[:max_sections], cited_refs


In [7]:
EXTRACTION_PROMPT = """You are a scientific paper analyst. Extract research entities from the paper below.

Title: {title}
Abstract: {abstract}

Extract ALL of the following. For every entity assign a confidence score (0.0-1.0):

- paper_summary      : 2-3 sentence plain-English summary of the paper's core contribution
- concepts_introduced: NEW concepts/frameworks/terms this paper PROPOSES (not pre-existing)
- concepts_applied   : EXISTING concepts or theories this paper USES
- methods            : Specific algorithms, models, or techniques (e.g. "BERT", "attention mechanism")
- datasets           : Datasets used for training or evaluation (e.g. "ImageNet", "SQuAD")
- topics             : Broad research areas. For each, include a parent_topic chain
                       (e.g. "Transformers" -> "Deep Learning" -> "Machine Learning")
- concept_relations  : Pairs of concepts from this paper that are closely related (RELATED_TO edges)
- method_extensions  : Method hierarchy pairs where child_method directly builds on parent_method
                       (e.g. child="LoRA", parent="PEFT")
- extends_papers     : Exact titles of prior papers this paper directly builds upon
- contradicts_papers : Exact titles of prior papers this paper disputes

Keep entity names concise (3-6 words). Add descriptions only when non-obvious.
Leave lists empty rather than guessing. Be conservative with confidence scores."""


def extract_entities_with_gemini(paper: dict) -> PaperExtraction:
    prompt = EXTRACTION_PROMPT.format(
        title=paper["title"],
        abstract=paper["abstract"],
    )
    return structured_llm.invoke(prompt)

In [8]:
def _embed(text: str) -> list:
    """
    Embed text via HuggingFace Inference API (BAAI/bge-base-en-v1.5, 768-dim).
    The API returns shape (seq_len, dim); we mean-pool to get the sentence vector.
    """
    arr = hf_client.feature_extraction(text, model=HF_EMBED_MODEL)
    arr = np.array(arr)
    if arr.ndim == 2:          # (seq_len, dim) → mean pool
        arr = arr.mean(axis=0)
    elif arr.ndim == 3:        # (batch, seq_len, dim) → squeeze batch + mean pool
        arr = arr[0].mean(axis=0)
    return arr.tolist()


def ingest_to_neo4j(
    paper: dict,
    extraction: PaperExtraction,
    sections: list[dict],
    cited_refs: list[dict],
):
    """
    Write one paper + all related nodes/edges into Neo4j.
    Each UNWIND is a separate execute_query call — avoids cartesian products.
    """
    aid = paper["arxiv_id"]

    def items(entities) -> list[dict]:
        return [e.model_dump() for e in entities]

    # ── 1. Paper node ──────────────────────────────────────────────────────
    driver.execute_query(
        """
        MERGE (p:Paper {arxiv_id: $arxiv_id})
        SET p.paper_id       = $arxiv_id,
            p.title          = $title,
            p.title_norm     = toLower(trim($title)),
            p.abstract       = $abstract,
            p.summary        = $summary,
            p.pdf_url        = $pdf_url,
            p.published_date = date($published),
            p.year           = $year,
            p.category       = $category,
            p.stub           = false
        """,
        arxiv_id=aid,
        title=paper["title"],
        abstract=paper["abstract"],
        summary=extraction.paper_summary,
        pdf_url=paper["pdf_url"],
        published=paper["published"],
        year=paper["year"],
        category=paper["category"],
        database_=NEO4J_DATABASE,
    )

    # ── 2. Authors + CO_AUTHORED ───────────────────────────────────────────
    if paper["authors"]:
        # Create Author nodes and AUTHORED_BY edges
        driver.execute_query(
            """
            MATCH (p:Paper {arxiv_id: $arxiv_id})
            UNWIND $authors AS author_name
              MERGE (a:Author {name_norm: toLower(trim(author_name))})
              ON CREATE SET a.author_id = randomUUID(), a.name = author_name
              MERGE (p)-[:AUTHORED_BY]->(a)
            """,
            arxiv_id=aid,
            authors=paper["authors"],
            database_=NEO4J_DATABASE,
        )
        # CO_AUTHORED edges between every pair of authors on this paper
        if len(paper["authors"]) > 1:
            driver.execute_query(
                """
                UNWIND $authors AS name_a
                UNWIND $authors AS name_b
                WITH name_a, name_b WHERE name_a < name_b
                MATCH (a:Author {name_norm: toLower(trim(name_a))})
                MATCH (b:Author {name_norm: toLower(trim(name_b))})
                MERGE (a)-[r:CO_AUTHORED]-(b)
                ON CREATE SET r.paper_count = 1
                ON MATCH  SET r.paper_count = r.paper_count + 1
                """,
                authors=paper["authors"],
                database_=NEO4J_DATABASE,
            )

    # ── 3. Concepts introduced ────────────────────────────────────────────
    if extraction.concepts_introduced:
        driver.execute_query(
            """
            MATCH (p:Paper {arxiv_id: $arxiv_id})
            UNWIND $items AS item
              MERGE (c:Concept {name_norm: toLower(trim(item.name))})
              ON CREATE SET c.concept_id = randomUUID(),
                            c.name = item.name,
                            c.description = item.description
              MERGE (p)-[r:INTRODUCES]->(c)
              SET r.source = 'llm', r.confidence = item.confidence
            """,
            arxiv_id=aid,
            items=items(extraction.concepts_introduced),
            database_=NEO4J_DATABASE,
        )

    # ── 4. Concepts applied ───────────────────────────────────────────────
    if extraction.concepts_applied:
        driver.execute_query(
            """
            MATCH (p:Paper {arxiv_id: $arxiv_id})
            UNWIND $items AS item
              MERGE (c:Concept {name_norm: toLower(trim(item.name))})
              ON CREATE SET c.concept_id = randomUUID(),
                            c.name = item.name,
                            c.description = item.description
              MERGE (p)-[r:APPLIES]->(c)
              SET r.source = 'llm', r.confidence = item.confidence
            """,
            arxiv_id=aid,
            items=items(extraction.concepts_applied),
            database_=NEO4J_DATABASE,
        )

    # ── 5. Methods ────────────────────────────────────────────────────────
    if extraction.methods:
        driver.execute_query(
            """
            MATCH (p:Paper {arxiv_id: $arxiv_id})
            UNWIND $items AS item
              MERGE (m:Method {name_norm: toLower(trim(item.name))})
              ON CREATE SET m.method_id = randomUUID(),
                            m.name = item.name,
                            m.description = item.description
              MERGE (p)-[r:USES_METHOD]->(m)
              SET r.source = 'llm', r.confidence = item.confidence
            """,
            arxiv_id=aid,
            items=items(extraction.methods),
            database_=NEO4J_DATABASE,
        )

    # ── 6. Datasets ───────────────────────────────────────────────────────
    if extraction.datasets:
        driver.execute_query(
            """
            MATCH (p:Paper {arxiv_id: $arxiv_id})
            UNWIND $items AS item
              MERGE (d:Dataset {name_norm: toLower(trim(item.name))})
              ON CREATE SET d.dataset_id = randomUUID(),
                            d.name = item.name,
                            d.description = item.description
              MERGE (p)-[r:USES_DATASET]->(d)
              SET r.source = 'llm', r.confidence = item.confidence
            """,
            arxiv_id=aid,
            items=items(extraction.datasets),
            database_=NEO4J_DATABASE,
        )

    # ── 7. Topics + hierarchy ─────────────────────────────────────────────
    if extraction.topics:
        topic_dicts = [t.model_dump() for t in extraction.topics]
        # Paper → Topic
        driver.execute_query(
            """
            MATCH (p:Paper {arxiv_id: $arxiv_id})
            UNWIND $topics AS t
              MERGE (tn:Topic {name_norm: toLower(trim(t.name))})
              ON CREATE SET tn.topic_id = randomUUID(),
                            tn.name = t.name,
                            tn.description = t.description
              MERGE (p)-[r:BELONGS_TO]->(tn)
              SET r.source = 'llm', r.confidence = t.confidence
            """,
            arxiv_id=aid,
            topics=topic_dicts,
            database_=NEO4J_DATABASE,
        )
        # Topic → Topic hierarchy (child BELONGS_TO parent)
        driver.execute_query(
            """
            UNWIND $topics AS t
              WITH t WHERE t.parent_topic IS NOT NULL AND t.parent_topic <> ''
              MERGE (child:Topic  {name_norm: toLower(trim(t.name))})
              MERGE (parent:Topic {name_norm: toLower(trim(t.parent_topic))})
              ON CREATE SET parent.topic_id = randomUUID(), parent.name = t.parent_topic
              MERGE (child)-[:BELONGS_TO]->(parent)
            """,
            topics=topic_dicts,
            database_=NEO4J_DATABASE,
        )

    # ── 8. Concept RELATED_TO Concept ────────────────────────────────────
    if extraction.concept_relations:
        driver.execute_query(
            """
            UNWIND $pairs AS pair
              MERGE (a:Concept {name_norm: toLower(trim(pair.concept_a))})
              ON CREATE SET a.concept_id = randomUUID(), a.name = pair.concept_a
              MERGE (b:Concept {name_norm: toLower(trim(pair.concept_b))})
              ON CREATE SET b.concept_id = randomUUID(), b.name = pair.concept_b
              MERGE (a)-[r:RELATED_TO]->(b)
              SET r.source = 'llm', r.confidence = pair.confidence
            """,
            pairs=[p.model_dump() for p in extraction.concept_relations],
            database_=NEO4J_DATABASE,
        )

    # ── 9. Method EXTENDS Method ──────────────────────────────────────────
    if extraction.method_extensions:
        driver.execute_query(
            """
            UNWIND $exts AS ext
              MERGE (child:Method  {name_norm: toLower(trim(ext.child_method))})
              ON CREATE SET child.method_id = randomUUID(), child.name = ext.child_method
              MERGE (parent:Method {name_norm: toLower(trim(ext.parent_method))})
              ON CREATE SET parent.method_id = randomUUID(), parent.name = ext.parent_method
              MERGE (child)-[r:EXTENDS]->(parent)
              SET r.source = 'llm', r.confidence = ext.confidence
            """,
            exts=[e.model_dump() for e in extraction.method_extensions],
            database_=NEO4J_DATABASE,
        )

    # ── 10. Paper EXTENDS Paper ───────────────────────────────────────────
    if extraction.extends_papers:
        driver.execute_query(
            """
            MATCH (p:Paper {arxiv_id: $arxiv_id})
            UNWIND $titles AS t
              MERGE (p2:Paper {title_norm: toLower(trim(t))})
              ON CREATE SET p2.title = t, p2.stub = true
              MERGE (p)-[r:EXTENDS]->(p2)
              SET r.source = 'llm'
            """,
            arxiv_id=aid,
            titles=extraction.extends_papers,
            database_=NEO4J_DATABASE,
        )

    # ── 11. Paper CONTRADICTS Paper ───────────────────────────────────────
    if extraction.contradicts_papers:
        driver.execute_query(
            """
            MATCH (p:Paper {arxiv_id: $arxiv_id})
            UNWIND $titles AS t
              MERGE (p2:Paper {title_norm: toLower(trim(t))})
              ON CREATE SET p2.title = t, p2.stub = true
              MERGE (p)-[r:CONTRADICTS]->(p2)
              SET r.source = 'llm'
            """,
            arxiv_id=aid,
            titles=extraction.contradicts_papers,
            database_=NEO4J_DATABASE,
        )

        # ── 12. Paper CITES Paper (from PDF bibliography) ────────────────────────────
    if cited_refs:
        arxiv_refs = [r for r in cited_refs if r.get("arxiv_id")]
        title_refs = [r for r in cited_refs if not r.get("arxiv_id") and r.get("title")]

        # 12a. arxiv-ID-keyed stubs (primary dedup key = arxiv_id)
        if arxiv_refs:
            driver.execute_query(
                """
                MATCH (p:Paper {arxiv_id: $arxiv_id})
                UNWIND $refs AS ref
                  MERGE (p2:Paper {arxiv_id: ref.arxiv_id})
                  ON CREATE SET p2.stub     = true,
                                p2.paper_id = ref.arxiv_id,
                                p2.title    = ref.title
                  MERGE (p)-[:CITES]->(p2)
                """,
                arxiv_id=aid,
                refs=arxiv_refs,
                database_=NEO4J_DATABASE,
            )

        # 12b. title-only stubs (fallback dedup key = title_norm)
        if title_refs:
            driver.execute_query(
                """
                MATCH (p:Paper {arxiv_id: $arxiv_id})
                UNWIND $refs AS ref
                  MERGE (p2:Paper {title_norm: toLower(trim(ref.title))})
                  ON CREATE SET p2.paper_id = randomUUID(),
                                p2.title    = ref.title,
                                p2.stub     = true
                  MERGE (p)-[:CITES]->(p2)
                """,
                arxiv_id=aid,
                refs=title_refs,
                database_=NEO4J_DATABASE,
            )

    # ── 13. Sections + embeddings + NEXT_SECTION chain ────────────────────
    section_ids = []
    for sec in sections:
        section_id = f"{aid}_s{sec['order']}"
        section_ids.append(section_id)

        # Compute embedding
        try:
            embedding = _embed(sec["content"])
        except Exception as e:
            print(f"          [warn] embedding failed for {section_id}: {e}")
            embedding = None

        driver.execute_query(
            """
            MERGE (s:Section {section_id: $section_id})
            SET s.title         = $title,
                s.content       = $content,
                s.section_order = $order,
                s.embedding     = $embedding
            WITH s
            MATCH (p:Paper {arxiv_id: $arxiv_id})
            MERGE (p)-[:HAS_SECTION]->(s)
            """,
            section_id=section_id,
            title=sec["title"],
            content=sec["content"],
            order=sec["order"],
            arxiv_id=aid,
            embedding=embedding,
            database_=NEO4J_DATABASE,
        )

    # NEXT_SECTION linked-list edges
    for i in range(len(section_ids) - 1):
        driver.execute_query(
            """
            MATCH (a:Section {section_id: $sid_a})
            MATCH (b:Section {section_id: $sid_b})
            MERGE (a)-[:NEXT_SECTION]->(b)
            """,
            sid_a=section_ids[i],
            sid_b=section_ids[i + 1],
            database_=NEO4J_DATABASE,
        )

    print(f"[+] Done: {aid}")

In [9]:
def run_pipeline(query: str, max_results: int = 2, skip_pdf: bool = False):
    """
    Args:
        query       : ArXiv search query (topic name or paper title)
        max_results : How many papers to ingest
        skip_pdf    : Skip PDF download — no Section nodes, no CITES edges (much faster)
    """
    print(f"\n{'='*60}\n  PaperGraph: '{query}'\n{'='*60}\n")

    papers = fetch_arxiv_papers(query, max_results=max_results)
    print(f"\nFetched {len(papers)} paper(s).\n")

    for i, paper in enumerate(papers, 1):
        print(f"[{i}/{len(papers)}] {paper['title'][:65]}")
        print(f"        arxiv_id : {paper['arxiv_id']}")

        # Entity extraction via Gemini
        print("        Extracting entities...")
        extraction = extract_entities_with_gemini(paper)
        print(f"          summary              : {extraction.paper_summary[:60]}...")
        print(f"          concepts_introduced  : {len(extraction.concepts_introduced)}")
        print(f"          concepts_applied     : {len(extraction.concepts_applied)}")
        print(f"          methods              : {len(extraction.methods)}")
        print(f"          datasets             : {len(extraction.datasets)}")
        print(f"          topics               : {len(extraction.topics)}")
        print(f"          concept_relations    : {len(extraction.concept_relations)}")
        print(f"          method_extensions    : {len(extraction.method_extensions)}")
        print(f"          extends_papers       : {len(extraction.extends_papers)}")
        print(f"          contradicts_papers   : {len(extraction.contradicts_papers)}")

        # PDF download → sections + citations
        sections, cited_refs = [], []
        if not skip_pdf:
            try:
                print("        Downloading PDF...")
                pdf_bytes = download_pdf(paper["pdf_url"])
                sections, cited_refs = parse_sections(pdf_bytes)
                n_arxiv = sum(1 for r in cited_refs if r.get("arxiv_id"))
                n_title = sum(1 for r in cited_refs if not r.get("arxiv_id"))
                print(f"          sections   : {len(sections)}")
                print(f"          citations  : {len(cited_refs)} total  "
                      f"({n_arxiv} with arxiv ID, {n_title} title-only stubs)")
            except Exception as e:
                print(f"          PDF failed ({e}) — skipping sections/citations")

        # Ingest everything into Neo4j
        print("        Ingesting into Neo4j...")
        ingest_to_neo4j(paper, extraction, sections, cited_refs)
        print(f"        Sleeping 2s...\n")
        time.sleep(2)

    print(f"\n=== Pipeline complete — {len(papers)} paper(s) ingested ===")

In [10]:
run_pipeline(
    query="Agentic AI Multi-Agent Systems",
    max_results=2,
    skip_pdf=False,
)


  PaperGraph: 'Agentic AI Multi-Agent Systems'

[*] Searching ArXiv for: Agentic AI Multi-Agent Systems
  [2203.08975] A Survey of Multi-Agent Deep Reinforcement Learning with Communication
  [1311.5108] A Methodology to Engineer and Validate Dynamic Multi-level Multi-agent

Fetched 2 paper(s).

[1/2] A Survey of Multi-Agent Deep Reinforcement Learning with Communic
        arxiv_id : 2203.08975
        Extracting entities...
          summary              : This paper surveys recent advancements in Multi-Agent Deep R...
          concepts_introduced  : 1
          concepts_applied     : 4
          methods              : 0
          datasets             : 0
          topics               : 4
          concept_relations    : 3
          method_extensions    : 0
          extends_papers       : 0
          contradicts_papers   : 0
          sections   : 15
          citations  : 32 total  (1 with arxiv ID, 31 title-only stubs)
        Ingesting into Neo4j...
[+] Done: 2203.08975
      

In [11]:
records, _, _ = driver.execute_query(
    "MATCH (p:Paper {stub: true}) RETURN p.arxiv_id AS id, p.title AS title ORDER BY id",
    database_=NEO4J_DATABASE,
)
print(f"{len(records)} stub paper(s) awaiting ingestion:")
for r in records:
    print(f"  {r['id'] or '(no id)':20s}  {(r['title'] or '')[:60]}")

50 stub paper(s) awaiting ingestion:
  2006.07869            
  (no id)               December 4-9, 2017, Long Beach, CA, USA, pages 6379–6390, 20
  (no id)               San Jose, California, USA, pages 709–715, 2004
  (no id)               Independent reinforcement learners in cooperative markov gam
  (no id)               Madison, Wisconsin, USA, pages 746–752, 1998
  (no id)               Stockholm, Sweden, July 10-15, 2018, pages 2085–2087, 2018. 
  (no id)               Long Beach, California, USA, volume 97 of Proceedings of Mac
  (no id)               Virtual Event, Austria, May 3-7, 2021, 2021
  (no id)               San Juan, Puerto Rico, May 2-4, 2016, Conference Track Proce
  (no id)               Conference Track Proceedings, 2018
  (no id)               New Orleans, LA, USA, May 6-9, 2019, 2019
  (no id)               Virtual Event, April 25-29, 2022, 2022
  (no id)               Online, July 5-10, 2020, pages 4427–4442, 2020
  (no id)               NeurIPS 2019, December